# Descomposición de la Variabilidad en la Liquidación de Siniestros a través de la Jerarquía Organizacional de una Aseguradora con PROC NESTED

## Resumen ejecutivo

Una aseguradora de daños y responsabilidad civil quiere saber **dónde** se origina la inconsistencia en la liquidación de siniestros: ¿la impulsan las diferencias entre regiones geográficas, entre oficinas de sucursal, entre ajustadores individuales, o simplemente el ruido siniestro a siniestro? Este cuaderno construye un libro sintético y completamente anidado de datos de siniestros de auto (Región > Sucursal > Ajustador > siniestro) y utiliza **PROC NESTED** para realizar un análisis de varianza de efectos aleatorios, estimando el componente de varianza aportado por cada nivel de la jerarquía y reportando cada uno como porcentaje del total. El resultado indica a la gerencia qué capa organizacional debe priorizarse para mejorar la consistencia en las liquidaciones.

## Fuentes de datos

Todos los datos se generan en línea mediante el primer DATA step — sin archivos externos, sin red. El diseño es **completamente anidado**: cada sucursal pertenece exactamente a una región, cada ajustador exactamente a una sucursal, y cada siniestro exactamente a un ajustador.

| Conjunto de datos | Filas | Variable | Tipo | Rol | Descripción |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (nivel 1, el más externo) | Región geográfica (5 regiones: NE, SE, MO, CO, SO) |
| | | `Branch` | char | CLASS (nivel 2, anidado en Region) | Código de oficina de sucursal (2 sucursales por región) |
| | | `Adjuster` | char | CLASS (nivel 3, anidado en Branch) | ID del ajustador de siniestros (2 ajustadores por sucursal) |
| | | `Settlement` | num | VAR (dependiente) | Indemnización pagada por el siniestro de auto, en USD |
| | | `CycleDays` | num | VAR (dependiente) | Días desde el primer aviso de siniestro hasta la liquidación |

Estructura sintética: 5 regiones x 2 sucursales x 2 ajustadores x 2 siniestros = 40 observaciones. Un efecto aleatorio de región, un efecto aleatorio de sucursal-dentro-de-región, un efecto aleatorio de ajustador-dentro-de-sucursal, y un residual a nivel de siniestro se superponen de forma aditiva mediante `rand('NORMAL', ...)` de modo que los componentes de varianza sean recuperables. El efecto de región se genera con la mayor desviación estándar (2,200) de modo que *dónde* se atiende un siniestro sea el factor dominante. Semilla fijada con `call streaminit(20260531)`. Un diseño compacto y completamente balanceado mantiene cada nivel de la jerarquía poblado con grados de libertad reales.

# Descomposición de la Variabilidad en la Liquidación de Siniestros con PROC NESTED

Cuando una aseguradora de daños y responsabilidad civil revisa cuánto paga para liquidar siniestros de auto, a menudo encuentra una amplia variación. Parte de esa variación es inevitable (cada accidente es diferente), pero parte de ella refleja factores **organizacionales** — una región opera con montos más altos que otra, una oficina de sucursal es más generosa, un ajustador individual es un valor atípico.

Los datos tienen una estructura estrictamente **anidada** (jerárquica):

```
Región  ->  Sucursal (anidada en Región)  ->  Ajustador (anidado en Sucursal)  ->  siniestros individuales
```

Una sucursal pertenece exactamente a una región y un ajustador exactamente a una sucursal, por lo que los factores están anidados en lugar de cruzados. `PROC NESTED` realiza un análisis de varianza de efectos aleatorios exactamente para este diseño y estima un **componente de varianza** en cada nivel, expresado como porcentaje del total. Ese desglose porcentual responde a la pregunta de negocio: *¿qué capa de la organización debemos priorizar para que las liquidaciones sean más consistentes?*

## Paso 1 — Generar un libro sintético y completamente anidado de siniestros

Simulamos 5 regiones, cada una con 2 sucursales, cada una con 2 ajustadores, cada uno atendiendo 2 siniestros (40 filas en total). Construimos la respuesta a partir de efectos aleatorios aditivos de modo que cada nivel aporte realmente varianza:

- un efecto de **región** (la mayor dispersión — las regiones difieren más),
- un efecto de **sucursal-dentro-de-región**,
- un efecto de **ajustador-dentro-de-sucursal**,
- y un **residual a nivel de siniestro** (el ruido dentro del ajustador).

`call streaminit` fija la semilla para reproducibilidad; cada efecto se genera con `rand('NORMAL', mean, sd)`. El efecto de región usa la desviación estándar más amplia, por lo que debería reclamar la mayor parte de la varianza total. También generamos una segunda respuesta, `CycleDays`, que comparte la misma jerarquía para poder demostrar más adelante el análisis multi-respuesta.

In [1]:
DATOS claims;
   LLAMAR streaminit(20260531);
   LONGITUD Region $2 Branch $6 Adjuster $9;

   /* Efecto aleatorio a nivel de región: las regiones difieren más. */
   HACER r = 1 HASTA 5;
      Region = SCAN('NE SE MO CO SO', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Sucursal anidada dentro de la región. */
      HACER b = 1 HASTA 2;
         Branch = catx('-', Region, PUT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Ajustador anidado dentro de la sucursal. */
         HACER a = 1 HASTA 2;
            Adjuster = catx('-', Branch, PUT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Siniestros individuales atendidos por este ajustador. */
            HACER claim = 1 HASTA 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               SI Settlement < 0 ENTONCES Settlement = 0;
               SI CycleDays  < 1 ENTONCES CycleDays  = 1;
               SALIDA;
            END;
         END;
      END;
   END;

   MANTENER Region Branch Adjuster Settlement CycleDays;
EJECUTAR;


NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Paso 2 — Ordenar según la jerarquía de anidamiento

`PROC NESTED` requiere que los datos de entrada estén **ordenados por las variables CLASS en el orden en que se listarán** — el factor más externo primero. Ordenamos por `Region Branch Adjuster` para que el procedimiento pueda recorrer la jerarquía correctamente.

In [2]:
PROCEDIMIENTO ORDENAR DATOS=claims;
   POR Region Branch Adjuster;
EJECUTAR;


NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Paso 3 — Análisis de componentes de varianza del monto de liquidación

El análisis central. Listamos las variables CLASS de la más externa a la más interna (`Region Branch Adjuster`); la replicación más interna — los siniestros individuales — se trata automáticamente como el término de error. `VAR Settlement` nombra la única variable de respuesta.

La sentencia `LABEL` provee un nombre de presentación más amigable para la respuesta en el encabezado de salida. La salida produce los coeficientes de cuadrados medios esperados, la tabla de análisis de varianza (con una prueba F de cada componente de varianza contra cero), y la estimación del **componente de varianza** en cada nivel junto con su **porcentaje del total**.

In [3]:
TÍTULO 'Componentes de Varianza de las Liquidaciones de Siniestros de Auto';
title2 'ANOVA de Efectos Aleatorios: Región / Sucursal / Ajustador';

PROCEDIMIENTO nested DATOS=claims;
   CLASE Region Branch Adjuster;
   VAR Settlement;
   ETIQUETA Region='Región' Branch='Sucursal' Adjuster='Ajustador'
            Settlement = 'Indemnización Pagada (USD)';
EJECUTAR;

                           Componentes de Varianza de las Liquidaciones de Siniestros de Auto                           
                               ANOVA de Efectos Aleatorios: Región / Sucursal / Ajustador                               


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnización Pagada (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Región                 4  62114122.126866          6.71      0.0304  Sucursal  15528530.531717  1651824.844989             54.4057      8.0000
Sucursal               5  11569658.859028          1.89      0.1829  Ajustador  2313931.771806   272682.828942              8.9813      4.0000
Ajustador             10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  2000069


NOTE: Option TITLE changed to Componentes de Varianza de las Liquidaciones de Siniestros de Auto.
NOTE: Option TITLE2 changed to ANOVA de Efectos Aleatorios: Región / Sucursal / Ajustador.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Paso 4 — Analizar dos respuestas a la vez

La gerencia también se preocupa por el **tiempo de ciclo** — cuánto tardan los siniestros en liquidarse. Agregamos `CycleDays` a la lista `VAR`. Con más de una variable dependiente, `PROC NESTED` reporta adicionalmente un **análisis de covariación** que muestra cómo covarían las dos respuestas en cada nivel de la jerarquía (por ejemplo, si las regiones que pagan más también liquidan más lento).

In [4]:
TÍTULO 'Componentes de Varianza de Liquidación y Tiempo de Ciclo';
title2 'Dos Variables de Respuesta en la Misma Jerarquía Anidada';

PROCEDIMIENTO nested DATOS=claims;
   CLASE Region Branch Adjuster;
   VAR Settlement CycleDays;
   ETIQUETA Region='Región' Branch='Sucursal' Adjuster='Ajustador'
            Settlement = 'Indemnización Pagada (USD)'
            CycleDays  = 'Días para Liquidar';
EJECUTAR;

                                Componentes de Varianza de Liquidación y Tiempo de Ciclo                                
                                Dos Variables de Respuesta en la Misma Jerarquía Anidada                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnización Pagada (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Región                 4  62114122.126866          6.71      0.0304  Sucursal  15528530.531717  1651824.844989             54.4057      8.0000
Sucursal               5  11569658.859028          1.89      0.1829  Ajustador  2313931.771806   272682.828942              8.9813      4.0000
Ajustador             10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  2000069


NOTE: Option TITLE changed to Componentes de Varianza de Liquidación y Tiempo de Ciclo.
NOTE: Option TITLE2 changed to Dos Variables de Respuesta en la Misma Jerarquía Anidada.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Paso 5 — Vista de solo varianza con la opción AOV

Para un resumen rápido de componentes de varianza en ambas respuestas, la opción `AOV` restringe la salida a las estadísticas de análisis de varianza y **suprime la sección de análisis de covariación**. Esta es la vista compacta que un analista de portafolio revisaría cuando solo necesita el desglose de varianza por nivel para cada respuesta, no la covariación entre respuestas.

In [5]:
TÍTULO 'Solo Componentes de Varianza (AOV)';

PROCEDIMIENTO nested DATOS=claims aov;
   CLASE Region Branch Adjuster;
   VAR Settlement CycleDays;
   ETIQUETA Region='Región' Branch='Sucursal' Adjuster='Ajustador'
            Settlement = 'Indemnización Pagada (USD)'
            CycleDays  = 'Días para Liquidar';
EJECUTAR;

TÍTULO;

                                           Solo Componentes de Varianza (AOV)                                           
                                Dos Variables de Respuesta en la Misma Jerarquía Anidada                                


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnización Pagada (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Región                 4  62114122.126866          6.71      0.0304  Sucursal  15528530.531717  1651824.844989             54.4057      8.0000
Sucursal               5  11569658.859028          1.89      0.1829  Ajustador  2313931.771806   272682.828942              8.9813      4.0000
Ajustador             10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  2000069


NOTE: Option TITLE changed to Solo Componentes de Varianza (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interpretación de los resultados

La columna **Porcentaje del Total** en la tabla de análisis de varianza es el dato principal. Leerla de arriba a abajo da la proporción de la variabilidad total de la liquidación atribuible a cada capa organizacional. Para el monto de liquidación, la ejecución produce:

- **Región — 54.4%** (Pr > F = 0.0304). Los datos se generaron con la mayor dispersión a nivel de región, y el análisis lo recupera: más de la mitad de toda la variabilidad en la liquidación está *entre* regiones, evidencia estadísticamente significativa de que *dónde* se atiende un siniestro es el factor dominante.
- **Sucursal (dentro de Región) — 9.0%** (Pr > F = 0.18). Una porción adicional modesta al descender de la región a las oficinas de sucursal individuales; no significativa aquí.
- **Ajustador (dentro de Sucursal) — 3.7%** (Pr > F = 0.33). Poca variación es atribuible al ajustador individual en este libro de negocio.
- **Error — 32.9%.** El ruido residual siniestro a siniestro dentro de un ajustador; esta es la variación irreducible que ninguna palanca organizacional puede eliminar.

Cada nivel también lleva una **prueba F (Pr > F)** de la hipótesis nula de que su componente de varianza es cero. El p-valor pequeño de Región (0.0304) es evidencia estadística de diferencias sistemáticas genuinas entre regiones, no azar del muestreo.

La respuesta de tiempo de ciclo cuenta una historia paralela: **Región representa el 45.8%** de la variación en días para liquidar (Pr > F = 0.0448, nuevamente significativo), con Sucursal y Ajustador contribuyendo participaciones de un solo dígito y el residual llevando el 40.1%. Así que tanto *cuánto* paga una aseguradora como *cuánto tarda* están gobernados, ante todo, por la región.

La ejecución con dos respuestas agrega un **análisis de covariación**. Aquí el nivel de Región impulsa los productos cruzados, y el **coeficiente de correlación global es -0.36** — una relación negativa: las regiones que pagan liquidaciones más grandes tienden a *cerrarlas más rápido*, no más lento. Ese es un hallazgo útil y no obvio — las regiones costosas no son las lentas.

**Conclusión de negocio:** dado que la Región domina el desglose de porcentaje del total para ambas respuestas, la aseguradora debería estandarizar primero las pautas de liquidación y los límites de autoridad *entre regiones* — ahí es donde vive la inconsistencia más grande y estadísticamente significativa — antes de invertir en intervenciones a nivel de sucursal o ajustador. La correlación negativa entre liquidación y tiempo de ciclo significa que no existe una única región que sea a la vez la más costosa y la más lenta, por lo que ambos problemas pueden abordarse con programas separados y focalizados por región. PROC NESTED convierte una sensación vaga de "nuestras liquidaciones son inconsistentes" en una atribución accionable, capa por capa, de dónde vive esa inconsistencia.